# Bank Customer Churn Prediction: Data Cleaning

**Project:** End-to-End Customer Churn Prediction for Financial Services  
**Phase:** 1 of 6 - Data Cleaning & Preparation  
**Author:** Sunmi  
**Dataset:** Bank Customer Churn (10,002 initial records)

---

## Executive Summary

This notebook documents the data cleaning process for a bank customer churn prediction project. The dataset contains information about 10,000+ bank customers.The target variable indicates whether a customer has exited (churned) from the bank.

**Key Cleaning Operations:**
- Removed 2 duplicate customer records
- Handled missing values across 4 features using statistical imputation
- Standardized data types for modeling compatibility
- Preserved data integrity while maintaining business logic

**Final Output:** A clean dataset of 10,000 unique customers ready for exploratory data analysis and feature engineering.

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Data Loading & Initial Inspection](#2-data-loading--initial-inspection)
3. [Handling Duplicate Records](#3-handling-duplicate-records)
4. [Missing Value Analysis](#4-missing-value-analysis)
5. [Missing Value Imputation](#5-missing-value-imputation)
   - 5.1 [Numeric Features (Age, HasCrCard, IsActiveMember)](#51-numeric-features)
   - 5.2 [Categorical Features (Geography)](#52-categorical-features)
6. [Data Type Standardization](#6-data-type-standardization)
7. [Final Data Validation](#7-final-data-validation)
8. [Export Clean Dataset](#8-export-clean-dataset)
9. [Cleaning Summary & Next Steps](#9-cleaning-summary--next-steps)

---
## 1. Environment Setup

We begin by importing the necessary Python libraries for data manipulation and analysis. Warnings are suppressed to maintain clean output throughout the notebook.

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

---
## 2. Data Loading & Initial Inspection

The raw dataset is loaded from CSV format. This initial inspection helps us understand the data structure, identify potential quality issues, and plan our cleaning strategy.

In [2]:
df=pd.read_csv(r"c:\Users\HP USER\Downloads\Bank customer churn_full project\bank_customer_churn_dataa.csv")

In [3]:
df.shape

(10002, 12)

The dataset contains **10,002 rows** (customer records) and **12 columns** (features). This is our starting point before any cleaning operations.

In [4]:
df.head(20)

,CustomerId,Surname,CreditScore,Geography,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15634602,Hargrave,619,France,42.00,2,0.00,1,1.0,1.0,101348.88,1
1,15647311,Hill,608,Spain,41.00,1,83807.86,1,0.0,1.0,112542.58,0
2,15619304,Onio,502,France,42.00,8,159660.80,3,1.0,0.0,113931.57,1
3,15701354,Boni,699,France,39.00,1,0.00,2,0.0,0.0,93826.63,0
4,15737888,Mitchell,850,Spain,43.00,2,125510.82,1,NaN,1.0,79084.10,0
5,15574012,Chu,645,Spain,44.00,8,113755.78,2,1.0,0.0,149756.71,1
6,15592531,Bartlett,822,NaN,50.00,7,0.00,2,1.0,1.0,10062.80,0
7,15656148,Obinna,376,Germany,29.00,4,115046.74,4,1.0,0.0,119346.88,1
8,15792365,He,501,France,44.00,4,142051.07,2,0.0,NaN,74940.50,0
9,15592389,H?,684,France,NaN,2,134603.88,1,1.0,1.0,71725.73,0


**Initial Observations:**
- Missing values visible in `Geography`, `Age`, `HasCrCard`, and `IsActiveMember`
- The `Surname` column contains some corrupted entries (e.g., "H?"), but this won't affect modeling as Surname is not a predictive feature
- Numeric features like `Age` and binary flags are currently stored as floats (likely due to missing values)

---
## 3. Handling Duplicate Records

Duplicate customer records can skew our analysis and model training. We identify and remove exact duplicates across all columns to ensure each customer appears only once in our dataset.

**Business Context:** In banking datasets, duplicates may arise from data entry errors, system migrations, or multiple account snapshots. Removing them ensures our churn analysis reflects unique customer behaviors rather than repeated observations.

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 10000
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerId       10000 non-null  int64  
 1   Surname          10000 non-null  object 
 2   CreditScore      10000 non-null  int64  
 3   Geography        9999 non-null   object 
 4   Age              9999 non-null   float64
 5   Tenure           10000 non-null  int64  
 6   Balance          10000 non-null  float64
 7   NumOfProducts    10000 non-null  int64  
 8   HasCrCard        9999 non-null   float64
 9   IsActiveMember   9999 non-null   float64
 10  EstimatedSalary  10000 non-null  float64
 11  Exited           10000 non-null  int64  
dtypes: float64(5), int64(5), object(2)
memory usage: 1015.6+ KB


**Result:** The dataset now contains **10,000 unique records** (reduced from 10,002), indicating that **2 duplicate rows** were successfully removed.

**Missing Value Summary (Before Imputation):**
- `Geography`: 1 missing value
- `Age`: 1 missing value
- `HasCrCard`: 1 missing value
- `IsActiveMember`: 1 missing value

These missing values represent only 0.01% of the data for each affected feature, suggesting high overall data quality.

---
## 4. Missing Value Analysis

Before applying any imputation strategy, let's examine the current state of the data more closely. Understanding the distribution and characteristics of our features helps inform the best approach for handling missing values.

In [7]:
df.head()

,CustomerId,Surname,CreditScore,Geography,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15634602,Hargrave,619,France,42.0,2,0.00,1,1.0,1.0,101348.88,1
1,15647311,Hill,608,Spain,41.0,1,83807.86,1,0.0,1.0,112542.58,0
2,15619304,Onio,502,France,42.0,8,159660.80,3,1.0,0.0,113931.57,1
3,15701354,Boni,699,France,39.0,1,0.00,2,0.0,0.0,93826.63,0
4,15737888,Mitchell,850,Spain,43.0,2,125510.82,1,NaN,1.0,79084.10,0


**Imputation Strategy:**

Given the minimal number of missing values (only 1 per affected feature), we'll use statistical imputation methods that preserve the distribution characteristics of each feature:

1. **Numeric Features (Age, HasCrCard, IsActiveMember):** Use **median imputation** rather than mean to reduce the impact of potential outliers
2. **Categorical Features (Geography):** Use **mode imputation** to fill with the most frequent country

This approach maintains data integrity while avoiding information loss from row deletion.

---
## 5. Missing Value Imputation

### 5.1 Numeric Features

For `Age`, `HasCrCard`, and `IsActiveMember`, we apply median imputation. The median is preferred over the mean because:
- It's robust to outliers and skewed distributions
- It preserves the central tendency of the data
- For binary features (HasCrCard, IsActiveMember), the median naturally falls on 0 or 1

#### Age Imputation

In [8]:
df['Age']=df['Age'].fillna(df['Age'].median())

The missing age value has been replaced with the median age of all customers. This ensures the imputed value represents a typical customer age without being influenced by unusually young or old outliers.

#### HasCrCard Imputation

In [9]:
df['HasCrCard']=df['HasCrCard'].fillna(df['HasCrCard'].median())

The missing credit card ownership status has been imputed with the median value (0 or 1), representing the most common status among customers.

#### IsActiveMember Imputation

In [10]:
df['IsActiveMember']=df['IsActiveMember'].fillna(df['IsActiveMember'].median())

The missing active membership status has been filled using the median, ensuring the imputed value aligns with typical customer behavior in the dataset.

### 5.2 Categorical Features

#### Geography Imputation

For the categorical `Geography` feature, we use mode imputation - filling the missing value with the most frequently occurring country. This preserves the distribution of customers across geographic regions.

In [11]:
df['Geography']=df['Geography'].fillna(df['Geography'].mode()[0])

The missing geography value has been replaced with the most common country in our dataset, maintaining the representative distribution of our customer base across France, Germany, and Spain.

---
## 6. Data Type Standardization

After imputation, we convert numeric features to their appropriate integer data types. This is essential for:
- **Model compatibility:** Many ML algorithms expect integer inputs for categorical/binary features
- **Memory efficiency:** Integer types consume less memory than floats
- **Data integrity:** Age and binary flags are conceptually integers, not decimals

#### Age Conversion

In [12]:
df['Age']=df['Age'].astype(int)

Age is now stored as integers (e.g., 42, 38, 29) rather than floats (42.0, 38.0, 29.0).

#### HasCrCard Conversion

In [13]:
df['HasCrCard']=df['HasCrCard'].astype(int)

Credit card ownership is now represented as pure binary integers: 0 (no card) or 1 (has card).

#### IsActiveMember Conversion

In [14]:
df['IsActiveMember']=df['IsActiveMember'].astype(int)

Active membership status is now stored as integers: 0 (inactive) or 1 (active).

---
## 7. Final Data Validation

Let's verify that our cleaning operations were successful by examining the final state of the dataset.

In [15]:
df.head(20)

,CustomerId,Surname,CreditScore,Geography,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15634602,Hargrave,619,France,42,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,41,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,42,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,39,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,43,2,125510.82,1,1,1,79084.10,0
5,15574012,Chu,645,Spain,44,8,113755.78,2,1,0,149756.71,1
6,15592531,Bartlett,822,France,50,7,0.00,2,1,1,10062.80,0
7,15656148,Obinna,376,Germany,29,4,115046.74,4,1,0,119346.88,1
8,15792365,He,501,France,44,4,142051.07,2,0,1,74940.50,0
9,15592389,H?,684,France,37,2,134603.88,1,1,1,71725.73,0


**Validation Checklist:**
-  No missing values visible (previously NaN values are now imputed)
-  Age, HasCrCard, and IsActiveMember are displayed as integers (no decimal points)
-  Geography contains valid country names (no NaN)
-  All 12 features are present and properly formatted
-  Data types are appropriate for each feature's semantic meaning

**Note:** The Surname column still contains some corrupted entries (e.g., "H?" in row 9). This is intentional - since Surname is not used as a modeling feature, we've chosen not to clean it, focusing our efforts on features that will impact churn prediction.

---
## 8. Export Clean Dataset

The cleaned dataset is now ready for the next phase of the project. We export it to CSV format for use in exploratory data analysis and subsequent modeling phases.

In [16]:
df.to_csv('bank_customer_churn_cleaned.csv', index=False)

**Output:** `bank_customer_churn_cleaned.csv`  
**Records:** 10,000 unique customers  
**Features:** 12 columns (all complete, no missing values)

---
## 9. Cleaning Summary & Next Steps

### Data Cleaning Summary

This notebook successfully transformed a raw banking dataset into a clean, analysis-ready format through the following operations:

**1. Duplicate Removal**
- Identified and removed 2 duplicate customer records
- Reduced dataset from 10,002 to 10,000 unique customers

**2. Missing Value Treatment**
- **Age:** 1 missing value imputed using median (robust to outliers)
- **HasCrCard:** 1 missing value imputed using median (maintains binary distribution)
- **IsActiveMember:** 1 missing value imputed using median (preserves typical behavior)
- **Geography:** 1 missing value imputed using mode (most common country)

**3. Data Type Optimization**
- Converted Age from float to integer
- Converted HasCrCard from float to integer (proper binary encoding)
- Converted IsActiveMember from float to integer (proper binary encoding)

**4. Data Quality Decisions**
- Retained corrupted Surname entries: Pragmatic decision since this feature won't be used in modeling
- Preserved all other features without modification: No evidence of data quality issues requiring intervention

### Dataset Profile After Cleaning

| Metric | Value |
|--------|-------|
| **Total Records** | 10,000 |
| **Total Features** | 12 |
| **Missing Values** | 0 (100% complete) |
| **Duplicate Records** | 0 (100% unique) |
| **Target Variable** | Exited (1 = Churned, 0 = Retained) |
| **Data Quality Score** |  Excellent (99.99% original data retained) |

### Key Insights for Modeling

1. **Minimal Data Loss:** Only 4 values (0.003% of total dataset) required imputation, suggesting high-quality source data
2. **Balanced Approach:** Used statistical methods (median/mode) that preserve data distributions rather than introducing bias
3. **Ready for Analysis:** All features are now in appropriate formats for both exploratory analysis and machine learning algorithms

### Next Steps: Phase 2 - Exploratory Data Analysis

With a clean dataset in hand, the next phase will focus on:

1. **Understanding Customer Demographics**
   - Age distribution and its relationship with churn
   - Geographic patterns across France, Germany, and Spain
   - Tenure patterns and customer lifecycle analysis

2. **Analyzing Financial Behaviors**
   - Account balance distributions and impact on retention
   - Product ownership patterns (1-4 products per customer)
   - Salary levels and their correlation with churn

3. **Identifying Churn Drivers**
   - Credit card ownership vs. churn rate
   - Active vs. inactive member behaviors
   - Feature correlations and multicollinearity assessment

4. **Visualizing Key Patterns**
   - Distribution plots for continuous variables
   - Churn rates across categorical segments
   - Feature importance preliminary assessment

The cleaned dataset provides a solid foundation for uncovering actionable insights about customer churn behavior in the banking sector.

---

**End of Data Cleaning Phase**  
**Output File:** `bank_customer_churn_cleaned.csv`  